<a href="https://colab.research.google.com/github/Najaf-Ali12/LLM_Finetuning_techniques/blob/main/LLM_finetuning_using_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 7.5 MB/s eta 0:00:00


In [39]:
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [52]:
Model_name="unsloth/Llama-3.2-1B-Instruct"
dataset="medalpaca/medical_meadow_medical_flashcards"
OUTPUT_DIR="Llama3-medical-lora"
NUM_SAMPLES=5000

In [41]:
medical_dataset=load_dataset(dataset,split="train")
medical_dataset

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 33955
})

In [42]:
# Getting KnowHow of dataset from its first three examples
for i in range(3):
  print("Input: ",medical_dataset[i]["input"])
  print("Output:", medical_dataset[i]["output"])
  print("Instructions: ",medical_dataset[i]["instruction"])
  print("- -"*30)


Input:  What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?
Output: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.
Instructions:  Answer this question truthfully
- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -
Input:  What leads to genitourinary syndrome of menopause (atrophic vaginitis)?
Output: Low estradiol production leads to genitourinary syndrome of menopause (atrophic vaginitis).
Instructions:  Answer this question truthfully
- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -
Input:  What does low REM sleep latency and experiencing hallucinations/sleep paralysis suggest?
Output: Low REM sleep latency and experiencing hallucinations/sleep paralysis suggests narcolepsy.
Instructions:  Answer this question truthfully
- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -


In [47]:
# Converting the message in a format model was trained upon
def to_messages(ex):
    return {"messages": [
        {"role": "system",    "content": ex["instruction"]},
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]}
# filtering rows where output column is msising or output column has nothing in it.
medical_dataset = medical_dataset.filter(lambda x: x["output"] and len(x["output"].strip()) > 0)
medical_dataset=medical_dataset.shuffle(seed=42)
dataset=medical_dataset.map(to_messages, remove_columns=medical_dataset.column_names)
print(len(dataset))

Filter:   0%|          | 0/33553 [00:00<?, ? examples/s]

Map:   0%|          | 0/33553 [00:00<?, ? examples/s]

33553


In [48]:
dataset["messages"][0]

[{'content': 'Answer this question truthfully', 'role': 'system'},
 {'content': 'What type of skin rash is frequently observed in patients with sarcoidosis?',
  'role': 'user'},
 {'content': 'Erythema nodosum is a type of skin rash that is commonly found in patients with sarcoidosis.',
  'role': 'assistant'}]

TypeError: string indices must be integers, not 'str'

In [49]:
train_ds,test_ds=dataset.train_test_split(test_size=0.1,seed=42).values()
train_ds

Dataset({
    features: ['messages'],
    num_rows: 30197
})

In [50]:
test_ds

Dataset({
    features: ['messages'],
    num_rows: 3356
})

In [54]:
tokenizer = AutoTokenizer.from_pretrained(Model_name)

#• Many causal language models (like Qwen or GPT) do not have a built-in padding token because they generate text continuously. However, during training, batch processing requires
# all sentences to be the exact same length.
# This line says: "If there isn't a padding token, just use the End-of-Sentence (eos_token) as a spacer to fill out short text."
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    Model_name, torch_dtype=torch.float32, device_map="auto"
)
model.config.use_cache = False

#Loops through every single layer in the model and counts the total number of parameters (weights).
n_params = sum(p.numel() for p in model.parameters())

#Counts only the parameters where requires_grad=True. In deep learning, if a parameter requires a gradient, it means it will be actively updated/changed by the trainer.
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params/1e6:.0f}M | Trainable: {n_train/1e6:.0f}M ({100*n_train/n_params:.0f}%)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Total params: 1236M | Trainable: 1236M (100%)


In [58]:
def ask(question, model, max_new_tokens=200):
    messages = [{"role": "system", "content": "Answer this question truthfully"},
                {"role": "user",   "content": question}]

    # converts the list of messages into the specific string format the model was trained to
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_questions = [
    "What is the mechanism of action of metformin?",
    "What are the classic symptoms of diabetic ketoacidosis?",
]
for q in test_questions:
    print("Q:", q); print("A:", ask(q, model)); print("=" * 60)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the mechanism of action of metformin?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Metformin is a medication that is primarily used to treat type 2 diabetes. Its mechanism of action involves several steps:

1. **Inhibiting glucose production in the liver**: Metformin works by decreasing the production of glucose in the liver. It does this by inhibiting the enzyme **phosphofructokinase-1 (PFK-1)**, which is involved in the glycolytic pathway that converts glucose into glucose-6-phosphate. By inhibiting PFK-1, metformin reduces the liver's ability to produce glucose, which in turn reduces blood glucose levels.

2. **Increasing insulin sensitivity**: Metformin also increases insulin sensitivity in the body, making it easier for glucose to enter cells. This is achieved by activating **peroxisome proliferator-activated receptor gamma (PPAR-γ)**, a type of nuclear receptor that regulates glucose and lipid metabolism.

3. **Reducing glucose uptake in muscles**: Metformin also reduces
Q: What are the classic symptoms of diabetic ketoacidosis?
A: Diabetic ketoacidosis (DKA

# Finetuning with LoRA

In [60]:
from huggingface_hub import login
login()

In [61]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter_id = "NajafAli01/llama3.2-1B-medical-lora"  # this is where the adapters will be saved

config = PeftConfig.from_pretrained(adapter_id)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

lora_model = PeftModel.from_pretrained(model, adapter_id)
lora_model.eval()

ValueError: Can't find 'adapter_config.json' at 'NajafAli01/llama3.2-1B-medical-lora'